# 🏆 Sélection Intelligente des Fournisseurs par IA
## Pipeline Complet — Data Engineering · Machine Learning · Scoring Frontend-Ready

---

> **Auteur :** Équipe Innovation Achats  
> **Version :** 2.0 — Pipeline Complet  
> **Statut :** Production-Ready Demo  
> **Datasets :** TED (7 fichiers CSV), GPPD (multi-pays), KPI simulés  

---

Ce notebook constitue un **livrable professionnel autonome** couvrant l'intégralité du pipeline :
de la donnée brute jusqu'au modèle IA prêt à être consommé par un frontend web.


## 1. Contexte Métier

La sélection des fournisseurs est une décision stratégique multicritère impliquant :

| Critère | Description |
|---------|-------------|
| **Qualité** | Taux de défauts, conformité, certifications |
| **Coût / TCO** | Prix d'achat, coûts cachés, risques financiers |
| **Livraison (OTIF)** | On-Time In-Full, lead time, fiabilité |
| **Flexibilité** | Réactivité aux variations de volumes |
| **Innovation** | R&D, digitalisation, co-développement |
| **ESG** | Environnement, Social, Gouvernance |

### Sources de données mobilisées

1. **TED (Tenders Electronic Daily)** — Données d'appels d'offres européens (7 fichiers CSV relationnels)
2. **GPPD (Global Public Procurement Database)** — Contrats publics mondiaux (multi-pays)
3. **KPI simulés** — Indicateurs fournisseurs générés de façon cohérente et reproductible

### Vision Future

Le modèle entraîné alimentera une application web permettant à un acheteur de saisir ses besoins et d'obtenir un classement des meilleurs fournisseurs avec explications.


## 2. Objectifs du Notebook

1. Charger et fusionner intelligemment les 7 fichiers TED CSV
2. Charger et concaténer les fichiers GPPD multi-pays
3. Harmoniser TED + GPPD dans un schéma commun
4. Générer un dataset KPI fournisseurs simulé, cohérent et reproductible
5. Nettoyer : nulls, doublons, incohérences, valeurs aberrantes
6. Construire un dataset final orienté Machine Learning
7. Entraîner et comparer plusieurs modèles de classification
8. Interpréter le meilleur modèle (feature importance, SHAP optionnel)
9. Sauvegarder tous les artefacts pour un futur backend/frontend
10. Démontrer la logique de scoring via une simulation d'usage réel


## 3. Import des Bibliothèques

In [1]:
import os
import re
import json
import warnings
import logging
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from scipy import stats

# Visualisation
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch

# Machine Learning — Scikit-learn
from sklearn.model_selection import (
    train_test_split, StratifiedShuffleSplit, GroupShuffleSplit, cross_val_score
)
from sklearn.preprocessing import (
    StandardScaler, OneHotEncoder, LabelEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
import joblib

# XGBoost — fallback propre
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
    print("✅ XGBoost disponible")
except ImportError:
    HAS_XGB = False
    print("⚠️  XGBoost non installé — on continuera sans")

# LightGBM — fallback propre
try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
    print("✅ LightGBM disponible")
except ImportError:
    HAS_LGBM = False
    print("⚠️  LightGBM non installé — on continuera sans")

# SHAP — fallback propre
try:
    import shap
    HAS_SHAP = True
    print("✅ SHAP disponible")
except ImportError:
    HAS_SHAP = False
    print("⚠️  SHAP non installé — interprétabilité de base uniquement")

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)
np.random.seed(42)

print("\n✅ Tous les imports réussis.")
print(f"   NumPy  : {np.__version__}")
print(f"   Pandas : {pd.__version__}")
print(f"   Date   : {datetime.now().strftime('%Y-%m-%d %H:%M')}")


✅ XGBoost disponible
⚠️  LightGBM non installé — on continuera sans
⚠️  SHAP non installé — interprétabilité de base uniquement

✅ Tous les imports réussis.
   NumPy  : 2.1.1
   Pandas : 2.2.3
   Date   : 2026-04-02 23:50


## 4. Paramètres Globaux et Chemins

In [2]:
# ─── Répertoires ──────────────────────────────────────────────────────────────
TED_DIR    = Path("../docs/raw/ted")          # Dossier contenant les 7 fichiers TED CSV
GPPD_DIR   = Path("../docs/raw/gppd")         # Dossier contenant les fichiers GPPD (un par pays ou un seul)
OUTPUT_DIR = Path("../app/backend/artifacts") # Artefacts exportés pour le backend/frontend

# Création automatique des dossiers de sortie
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "figures").mkdir(exist_ok=True)

# ─── Options de développement ─────────────────────────────────────────────────
USE_SAMPLE    = True     # True = travailler sur un échantillon si données volumineuses
SAMPLE_N      = 50_000   # Nombre max de lignes si USE_SAMPLE = True
RANDOM_SEED   = 42

# ─── Fichiers TED attendus ────────────────────────────────────────────────────
TED_FILES_EXPECTED = [
    "awards.csv",
    "bids_details.csv",
    "bids_statistics.csv",
    "main.csv",
    "parties.csv",
    "tender_items.csv",
    "tender_lots.csv",
]

# ─── Schéma harmonisé cible (colonnes communes TED + GPPD) ────────────────────
HARMONIZED_SCHEMA = [
    "record_source",    # 'TED' ou 'GPPD'
    "tender_id",
    "lot_id",
    "bid_id",
    "buyer_id",
    "buyer_name",
    "buyer_country",
    "supplier_id",
    "supplier_name",
    "supplier_country",
    "cpv_code",
    "procedure_type",
    "estimated_price",
    "bid_price",
    "currency",
    "year",
    "num_bids",
    "is_winner",
]

# ─── Colonnes post-award à exclure (fuite de cible) ──────────────────────────
LEAKAGE_COLUMNS_PATTERNS = [
    "award_date", "contract_date", "signed_date", "decision_date",
    "award_value", "contract_value", "final_price", "awarded_price",
    "contract_url", "contract_id", "award_status",
    "lot_status",         # encode souvent le résultat
    "award_criterion_type",  # parfois post-décision
]

print("✅ Paramètres initialisés.")
print(f"   TED_DIR    : {TED_DIR.resolve()}")
print(f"   GPPD_DIR   : {GPPD_DIR.resolve()}")
print(f"   OUTPUT_DIR : {OUTPUT_DIR.resolve()}")
print(f"   USE_SAMPLE : {USE_SAMPLE} (max {SAMPLE_N:,} lignes)")


✅ Paramètres initialisés.
   TED_DIR    : C:\Users\hkmoh\Downloads\supplier_ai_package\supplier_ai_package\docs\raw\ted
   GPPD_DIR   : C:\Users\hkmoh\Downloads\supplier_ai_package\supplier_ai_package\docs\raw\gppd
   OUTPUT_DIR : C:\Users\hkmoh\Downloads\supplier_ai_package\supplier_ai_package\app\backend\artifacts
   USE_SAMPLE : True (max 50,000 lignes)


## 5. Fonctions Utilitaires

In [3]:
# ─── Normalisation des noms de colonnes ───────────────────────────────────────
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalise tous les noms de colonnes : lowercase, strip, _ pour espaces/tirets."""
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r'[\s\-]+', '_', regex=True)
        .str.replace(r'[^\w]', '', regex=True)
        .str.replace(r'_+', '_', regex=True)
        .str.strip('_')
    )
    # Déduplique les noms de colonnes
    seen = {}
    new_cols = []
    for col in df.columns:
        if col in seen:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
        else:
            seen[col] = 0
            new_cols.append(col)
    df.columns = new_cols
    return df

# ─── Synonymes de colonnes à harmoniser ───────────────────────────────────────
COLUMN_SYNONYMS = {
    # Identifiants fournisseur
    "supplier_id": ["bidder_id", "vendor_id", "party_id_bidder", "offeror_id"],
    "supplier_name": ["bidder_name", "vendor_name", "party_name_bidder"],
    # Identifiants acheteur
    "buyer_id": ["contracting_authority_id", "ca_id", "party_id_buyer"],
    "buyer_name": ["contracting_authority_name", "ca_name", "party_name_buyer"],
    # Prix
    "bid_price": ["bid_priceeur", "bid_priceusd", "offer_price", "lot_bid_price"],
    "estimated_price": ["tender_estimatedprice", "estimated_value", "lot_estimatedprice"],
    # Label
    "is_winner": ["bid_iswinning", "awarded", "isawarded", "winner"],
    # Autres
    "tender_id": ["tender_id", "procedure_id", "ocid"],
    "lot_id": ["lot_id", "lotid", "lotnumber"],
    "cpv_code": ["tender_maincpv", "cpv", "maincpv", "item_cpv"],
    "num_bids": ["bids_count", "number_of_bids", "num_offers"],
    "year": ["tender_year", "award_year", "publication_year"],
    "currency": ["bid_currency", "tender_currency"],
    "buyer_country": ["contracting_authority_country", "ca_country", "buyer_countryiso2"],
    "supplier_country": ["bidder_country", "vendor_country", "bidder_countryiso2"],
}

def apply_synonyms(df: pd.DataFrame, synonyms: dict = COLUMN_SYNONYMS) -> pd.DataFrame:
    """Renomme les colonnes selon le dictionnaire de synonymes (premier trouvé gagne)."""
    df = df.copy()
    rename_map = {}
    existing_targets = set(df.columns)
    for target, aliases in synonyms.items():
        if target in existing_targets:
            continue  # déjà bien nommé
        for alias in aliases:
            if alias in df.columns:
                rename_map[alias] = target
                existing_targets.add(target)
                break
    if rename_map:
        df = df.rename(columns=rename_map)
    return df

# ─── Chargement sécurisé d'un CSV ─────────────────────────────────────────────
def safe_read_csv(path: Path, **kwargs) -> pd.DataFrame | None:
    """Lit un CSV avec gestion d'erreur. Retourne None si fichier absent."""
    if not path.exists():
        print(f"   ⚠️  Fichier absent : {path}")
        return None
    try:
        df = pd.read_csv(path, low_memory=False, on_bad_lines='skip', **kwargs)
        df = normalize_columns(df)
        df = apply_synonyms(df)
        print(f"   ✅ {path.name:<30} → {df.shape[0]:>7,} lignes × {df.shape[1]:>3} colonnes")
        return df
    except Exception as e:
        print(f"   ❌ Erreur lecture {path.name} : {e}")
        return None

# ─── Rapport nulls ─────────────────────────────────────────────────────────────
def null_report(df: pd.DataFrame, label: str = "DataFrame") -> pd.DataFrame:
    """Retourne un DataFrame avec le taux de nulls par colonne."""
    report = pd.DataFrame({
        'missing_count': df.isnull().sum(),
        'missing_pct': (df.isnull().sum() / len(df) * 100).round(2)
    })
    report = report[report['missing_count'] > 0].sort_values('missing_pct', ascending=False)
    print(f"\n📊 Nulls [{label}] — {len(report)} colonnes avec valeurs manquantes sur {df.shape[1]}")
    return report

# ─── Suppression des colonnes fuite de cible ──────────────────────────────────
def remove_leakage_columns(df: pd.DataFrame, patterns: list = LEAKAGE_COLUMNS_PATTERNS,
                            keep_target: str = 'is_winner') -> tuple:
    """Identifie et retire les colonnes susceptibles de provoquer une fuite de cible."""
    cols_to_drop = []
    for col in df.columns:
        if col == keep_target:
            continue
        for pattern in patterns:
            if pattern.lower() in col.lower():
                cols_to_drop.append(col)
                break
    cols_to_drop = list(set(cols_to_drop))
    df_clean = df.drop(columns=cols_to_drop, errors='ignore')
    return df_clean, cols_to_drop

# ─── Winsorization (traitement outliers) ──────────────────────────────────────
def winsorize_col(series: pd.Series, lower_q: float = 0.01, upper_q: float = 0.99) -> pd.Series:
    """Capping par quantiles."""
    lo = series.quantile(lower_q)
    hi = series.quantile(upper_q)
    return series.clip(lower=lo, upper=hi)

# ─── Rapport shape ──────────────────────────────────────────────────────────
def shape_report(df: pd.DataFrame, name: str):
    print(f"   📐 {name}: {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

print("✅ Fonctions utilitaires chargées.")


✅ Fonctions utilitaires chargées.


## 6. Chargement des Données TED

Les 7 fichiers TED sont chargés indépendamment avec rapport de disponibilité.  
Si aucun fichier réel n'est trouvé, un dataset synthétique crédible est généré automatiquement.


In [4]:
print("=" * 65)
print("CHARGEMENT DES DONNÉES TED")
print("=" * 65)

ted_raw = {}
ted_found = []
ted_missing = []

for fname in TED_FILES_EXPECTED:
    path = TED_DIR / fname
    key = fname.replace(".csv", "")
    df = safe_read_csv(path)
    if df is not None:
        ted_raw[key] = df
        ted_found.append(fname)
    else:
        ted_missing.append(fname)

print(f"\n📦 Fichiers TED trouvés   : {len(ted_found)}")
print(f"   Fichiers TED manquants : {len(ted_missing)}")
if ted_missing:
    print(f"   Manquants : {', '.join(ted_missing)}")

TED_MODE = 'real' if len(ted_found) >= 2 else 'synthetic'
print(f"\n🔧 Mode TED : {'Données réelles' if TED_MODE == 'real' else 'Données synthétiques (aucun fichier TED détecté)'}")


CHARGEMENT DES DONNÉES TED
   ✅ awards.csv                     →     101 lignes ×  19 colonnes
   ✅ bids_details.csv               →     101 lignes ×   8 colonnes
   ✅ bids_statistics.csv            →     119 lignes ×   6 colonnes
   ✅ main.csv                       →     516 lignes ×  76 colonnes
   ✅ parties.csv                    →     659 lignes ×  14 colonnes
   ✅ tender_items.csv               →   1,741 lignes ×   7 colonnes
   ✅ tender_lots.csv                →   1,127 lignes ×   9 colonnes

📦 Fichiers TED trouvés   : 7
   Fichiers TED manquants : 0

🔧 Mode TED : Données réelles


## 7. Génération Synthétique TED (si fichiers absents)

In [5]:
def generate_synthetic_ted(n_tenders=3000, seed=42):
    """Génère un dataset TED synthétique réaliste de niveau 1 ligne = 1 offre fournisseur."""
    rng = np.random.default_rng(seed)
    
    countries = ['FR','DE','IT','ES','PL','NL','BE','SE','AT','PT',
                 'CZ','RO','HU','SK','FI','DK','GR','HR','BG','LT']
    cpv_codes = ['45000000','33000000','72000000','48000000','60000000',
                 '79000000','50000000','71000000','90000000','34000000',
                 '15000000','30000000','31000000','35000000','39000000']
    procedure_types = ['OPEN','RESTRICTED','NEGOTIATED','COMPETITIVE_DIALOGUE','DIRECT']
    
    records = []
    supplier_pool = [f"SUP_{i:05d}" for i in range(1, 501)]
    buyer_pool    = [f"BUY_{i:04d}" for i in range(1, 201)]
    
    tid = 0
    for _ in range(n_tenders):
        tid += 1
        tender_id = f"TED-{tid:06d}"
        n_lots = rng.integers(1, 4)
        buyer_id = rng.choice(buyer_pool)
        buyer_country = rng.choice(countries)
        cpv = rng.choice(cpv_codes)
        proc_type = rng.choice(procedure_types, p=[0.55,0.20,0.12,0.08,0.05])
        est_price_lot = rng.exponential(scale=250_000) + 10_000
        n_bids = rng.integers(2, 12)
        year = int(rng.integers(2018, 2024))
        
        for lot_n in range(1, n_lots + 1):
            lot_id = f"{tender_id}-L{lot_n}"
            bidders = rng.choice(supplier_pool, size=min(n_bids, len(supplier_pool)), replace=False)
            winner_idx = rng.integers(0, len(bidders))
            
            for j, sup_id in enumerate(bidders):
                price_noise = rng.normal(1.0, 0.15)
                bid_px = max(est_price_lot * price_noise, 1000)
                records.append({
                    'tender_id':   tender_id,
                    'lot_id':      lot_id,
                    'bid_id':      f"BID-{tid:06d}-L{lot_n}-{j+1:02d}",
                    'buyer_id':    buyer_id,
                    'buyer_name':  f"Authority_{buyer_id}",
                    'buyer_country': buyer_country,
                    'supplier_id': sup_id,
                    'supplier_name': f"Company_{sup_id}",
                    'supplier_country': rng.choice(countries),
                    'cpv_code':    cpv,
                    'procedure_type': proc_type,
                    'estimated_price': round(est_price_lot, 2),
                    'bid_price':   round(bid_px, 2),
                    'currency':    'EUR',
                    'year':        year,
                    'num_bids':    int(n_bids),
                    'is_winner':   int(j == winner_idx),
                    'record_source': 'TED',
                })
    
    df = pd.DataFrame(records)
    print(f"✅ Dataset TED synthétique généré : {df.shape}")
    print(f"   Tenders : {df['tender_id'].nunique():,} | Lots : {df['lot_id'].nunique():,}")
    print(f"   Gagnants : {df['is_winner'].sum():,} ({df['is_winner'].mean()*100:.1f}%)")
    return df

if TED_MODE == 'synthetic':
    ted_ml = generate_synthetic_ted(n_tenders=3000, seed=RANDOM_SEED)
    ted_ml['record_source'] = 'TED'
else:
    ted_ml = None  # sera construit à l'étape de fusion
    print("ℹ️  Données réelles détectées → fusion à l'étape suivante.")


ℹ️  Données réelles détectées → fusion à l'étape suivante.


## 8. Fusion des Fichiers TED (données réelles)

Stratégie de fusion professionnelle :
1. `bids_details` = table centrale (1 ligne = 1 offre)
2. Enrichissement depuis `main`, `tender_lots`, `bids_statistics`, `awards`, `parties`
3. Création du label `is_winner` depuis `awards`


In [6]:
def detect_common_key(df1: pd.DataFrame, df2: pd.DataFrame,
                      candidates: list = None) -> str | None:
    """Trouve la première colonne commune parmi les candidats."""
    if candidates is None:
        candidates = ['bid_id','lot_id','tender_id','supplier_id','buyer_id']
    c1, c2 = set(df1.columns), set(df2.columns)
    for c in candidates:
        if c in c1 and c in c2:
            return c
    common = c1 & c2
    return next(iter(common), None)

def smart_merge(base: pd.DataFrame, other: pd.DataFrame,
                name: str, key: str = None,
                how: str = 'left') -> pd.DataFrame:
    """Merge avec logging, contrôle de cardinalité et dédupliction."""
    if other is None:
        print(f"   ⚠️  {name} absent — merge ignoré")
        return base
    if key is None:
        key = detect_common_key(base, other)
    if key is None:
        print(f"   ⚠️  {name} — aucune clé commune détectée, merge ignoré")
        return base
    
    # Colonnes à ajouter (évite doublons de colonnes autre que la clé)
    new_cols = [c for c in other.columns if c not in base.columns or c == key]
    other_sub = other[new_cols].drop_duplicates(subset=[key], keep='first')
    
    before = len(base)
    merged = base.merge(other_sub, on=key, how=how, suffixes=('', f'_{name}'))
    after = len(merged)
    nulls_new = merged[[c for c in new_cols if c != key and c in merged.columns]].isnull().mean().mean()
    
    print(f"   🔗 + {name:<20} | clé={key} | lignes: {before:,} → {after:,} | nulls nouveaux: {nulls_new:.1%}")
    return merged

if TED_MODE == 'real' and ted_ml is None:
    print("=" * 65)
    print("FUSION TED — Données réelles")
    print("=" * 65)
    
    # ── Base centrale : bids_details ──────────────────────────────────────
    base_key = 'bids_details'
    if base_key in ted_raw:
        ted_ml = ted_raw[base_key].copy()
        print(f"\n✅ Base centrale : bids_details — {ted_ml.shape}")
    elif 'main' in ted_raw:
        ted_ml = ted_raw['main'].copy()
        print(f"\n⚠️  bids_details absent — utilisation de main comme base : {ted_ml.shape}")
    else:
        first_key = next(iter(ted_raw))
        ted_ml = ted_raw[first_key].copy()
        print(f"\n⚠️  Fallback sur premier fichier disponible : {first_key}")
    
    # ── Enrichissements ───────────────────────────────────────────────────
    # 1. main (contexte global tender)
    if 'main' in ted_raw:
        ted_ml = smart_merge(ted_ml, ted_raw['main'], 'main',
                             key=detect_common_key(ted_ml, ted_raw['main']))
    
    # 2. tender_lots (contexte lot + prix estimé)
    if 'tender_lots' in ted_raw:
        ted_ml = smart_merge(ted_ml, ted_raw['tender_lots'], 'tender_lots',
                             key=detect_common_key(ted_ml, ted_raw['tender_lots']))
    
    # 3. bids_statistics (enrichissement concurrence agrégé)
    if 'bids_statistics' in ted_raw:
        ted_ml = smart_merge(ted_ml, ted_raw['bids_statistics'], 'bids_statistics',
                             key=detect_common_key(ted_ml, ted_raw['bids_statistics']))
    
    # 4. parties (info acheteur / fournisseur)
    if 'parties' in ted_raw:
        ted_ml = smart_merge(ted_ml, ted_raw['parties'], 'parties',
                             key=detect_common_key(ted_ml, ted_raw['parties']))
    
    # 5. tender_items (agréger au niveau lot)
    if 'tender_items' in ted_raw:
        ti = ted_raw['tender_items']
        key_ti = detect_common_key(ted_ml, ti)
        if key_ti:
            ti_agg = ti.select_dtypes(include='number').groupby(
                ti[key_ti] if key_ti in ti.columns else ti.index
            ).mean().add_prefix('items_avg_').reset_index().rename(columns={key_ti: key_ti})
            ted_ml = smart_merge(ted_ml, ti_agg, 'tender_items_agg', key=key_ti)
    
    # 6. awards → label is_winner
    if 'awards' in ted_raw:
        aw = ted_raw['awards'].copy()
        winner_col_candidates = ['is_winner','awarded','bid_iswinning','isawarded','winner']
        win_col = next((c for c in winner_col_candidates if c in aw.columns), None)
        key_aw = detect_common_key(ted_ml, aw, ['bid_id','lot_id','supplier_id','tender_id'])
        if win_col and key_aw:
            aw_sub = aw[[key_aw, win_col]].rename(columns={win_col: 'is_winner_from_awards'})
            ted_ml = smart_merge(ted_ml, aw_sub, 'awards', key=key_aw)
            if 'is_winner' not in ted_ml.columns and 'is_winner_from_awards' in ted_ml.columns:
                ted_ml['is_winner'] = ted_ml['is_winner_from_awards']
        elif key_aw:
            # Award = tout ce qui se trouve dans la table awards est gagnant
            aw_ids = aw[key_aw].unique()
            if 'is_winner' not in ted_ml.columns:
                ted_ml['is_winner'] = ted_ml[key_aw].isin(aw_ids).astype(int)
            print(f"   ℹ️  is_winner déduit par présence dans awards (clé={key_aw})")
    
    # S'assurer que is_winner existe
    if 'is_winner' not in ted_ml.columns:
        print("   ⚠️  is_winner introuvable — label créé de façon aléatoire (1 par lot)")
        ted_ml['is_winner'] = 0
        for lid, grp in ted_ml.groupby(ted_ml.get('lot_id', ted_ml.index)):
            idx = grp.sample(1, random_state=42).index
            ted_ml.loc[idx, 'is_winner'] = 1
    
    ted_ml['record_source'] = 'TED'
    print(f"\n✅ ted_ml final : {ted_ml.shape}")
    print(f"   Gagnants : {ted_ml['is_winner'].sum():,} ({ted_ml['is_winner'].mean()*100:.1f}%)")
else:
    print("ℹ️  TED synthétique déjà généré — étape réelle ignorée.")

shape_report(ted_ml, "ted_ml")


FUSION TED — Données réelles

✅ Base centrale : bids_details — (101, 8)
   🔗 + main                 | clé=id | lignes: 101 → 101 | nulls nouveaux: 100.0%
   🔗 + tender_lots          | clé=tender_id | lignes: 101 → 101 | nulls nouveaux: 100.0%
   🔗 + bids_statistics      | clé=main_id | lignes: 101 → 101 | nulls nouveaux: 61.4%
   🔗 + parties              | clé=main_id | lignes: 101 → 101 | nulls nouveaux: 8.9%
   🔗 + tender_items_agg     | clé=tender_id | lignes: 101 → 101 | nulls nouveaux: 100.0%
   ℹ️  is_winner déduit par présence dans awards (clé=relatedlots)

✅ ted_ml final : (101, 103)
   Gagnants : 101 (100.0%)
   📐 ted_ml: 101 lignes × 103 colonnes


## 9. Chargement des Données GPPD (Multi-pays)

Le notebook accepte :
- Un seul fichier GPPD global
- Plusieurs fichiers GPPD séparés par pays (scan automatique du dossier `GPPD_DIR`)


In [ ]:
print("=" * 65)
print("CHARGEMENT DES DONNÉES GPPD")
print("=" * 65)

gppd_frames = []

# ─── Scan du dossier GPPD ────────────────────────────────────────────────────
gppd_csv_files = sorted(GPPD_DIR.glob("*.csv")) if GPPD_DIR.exists() else []

# Essayer aussi un fichier unique à la racine
single_gppd = Path("./gppd.csv")
if single_gppd.exists() and single_gppd not in gppd_csv_files:
    gppd_csv_files = [single_gppd] + list(gppd_csv_files)

print(f"   Fichiers GPPD trouvés : {len(gppd_csv_files)}")
for f in gppd_csv_files:
    print(f"     → {f}")

GPPD_MODE = 'real' if gppd_csv_files else 'synthetic'

if GPPD_MODE == 'real':
    for csv_path in gppd_csv_files:
        df = safe_read_csv(csv_path)
        if df is None:
            continue
        
        # Ajouter la colonne source_file
        df['source_file'] = csv_path.name
        
        # Déduire source_country
        if 'buyer_country' in df.columns:
            df['source_country'] = df['buyer_country'].fillna(
                csv_path.stem.split('_')[-1].upper()
            )
        elif 'source_country' not in df.columns:
            country_code = csv_path.stem.upper().replace('GPPD_','').replace('GPPD','MULTI')
            df['source_country'] = country_code
        
        gppd_frames.append(df)
    
    if gppd_frames:
        # Rapport colonnes par fichier
        all_cols = set()
        for df in gppd_frames:
            all_cols |= set(df.columns)
        print(f"\n   Colonnes union : {len(all_cols)}")
        
        # Colonnes présentes dans chaque fichier
        col_presence = {}
        for i, df in enumerate(gppd_frames):
            for col in all_cols:
                col_presence.setdefault(col, 0)
                if col in df.columns:
                    col_presence[col] += 1
        
        missing_cols = {k: v for k, v in col_presence.items() if v < len(gppd_frames)}
        if missing_cols:
            print(f"   Colonnes absentes dans certains pays : {len(missing_cols)}")
        
        gppd_raw_all = pd.concat(gppd_frames, ignore_index=True, sort=False)
        print(f"\n✅ GPPD concaténé : {gppd_raw_all.shape}")
    else:
        GPPD_MODE = 'synthetic'
        print("⚠️  Aucun fichier GPPD lisible — bascule en mode synthétique")
else:
    print("   Aucun fichier GPPD détecté → génération synthétique")


CHARGEMENT DES DONNÉES GPPD
   Fichiers GPPD trouvés : 11
     → ..\docs\raw\gppd\MX_DIB_2023.csv
     → ..\docs\raw\gppd\NL_DIB_2023.csv
     → ..\docs\raw\gppd\NO_DIB_2023.csv
     → ..\docs\raw\gppd\PL_DIB_2023.csv
     → ..\docs\raw\gppd\PT_DIB_2023.csv
     → ..\docs\raw\gppd\PY_DIB_2023.csv
     → ..\docs\raw\gppd\RO_DIB_2023.csv
     → ..\docs\raw\gppd\SE_DIB_2023.csv
     → ..\docs\raw\gppd\SI_DIB_2023.csv
     → ..\docs\raw\gppd\SK_DIB_2023.csv
     → ..\docs\raw\gppd\UY_DIB_2023.csv


## 10. Génération Synthétique GPPD (si fichiers absents)

In [ ]:
def generate_synthetic_gppd(n_records=5000, seed=42):
    """Génère un dataset GPPD synthétique multi-pays."""
    rng = np.random.default_rng(seed)
    
    countries = ['FR','DE','IT','ES','PL','NL','BE','SE','PT','CZ',
                 'RO','HU','SK','FI','DK','GR','HR','BG','LT','LV']
    cpv_codes = ['45000000','33000000','72000000','48000000','60000000',
                 '79000000','50000000','71000000','90000000','34000000']
    proc_types = ['OPEN','RESTRICTED','NEGOTIATED','DIRECT','AWARD_WO_PRIOR_PUB']
    
    supplier_pool = [f"GSUP_{i:05d}" for i in range(1, 601)]
    buyer_pool    = [f"GBUY_{i:04d}" for i in range(1, 251)]
    
    records = []
    for i in range(n_records):
        buyer_c   = rng.choice(countries)
        supplier_c = rng.choice(countries)
        est_px    = float(rng.exponential(200_000) + 5_000)
        bid_px    = est_px * float(rng.normal(1.0, 0.18))
        bid_px    = max(bid_px, 500)
        n_bids    = int(rng.integers(1, 15))
        records.append({
            'persistent_id':         f"GPPD-{i+1:07d}",
            'tender_id':             f"GPPD-T-{rng.integers(1, n_records//3+1):06d}",
            'lot_id':                f"GPPD-L-{rng.integers(1, n_records//2+1):06d}",
            'bid_id':                f"GPPD-B-{i+1:07d}",
            'buyer_id':              rng.choice(buyer_pool),
            'buyer_name':            f"Authority_{rng.choice(buyer_pool)}",
            'buyer_country':         buyer_c,
            'supplier_id':           rng.choice(supplier_pool),
            'supplier_name':         f"Company_{rng.choice(supplier_pool)}",
            'supplier_country':      supplier_c,
            'cpv_code':              rng.choice(cpv_codes),
            'procedure_type':        rng.choice(proc_types, p=[0.5,0.2,0.15,0.08,0.07]),
            'estimated_price':       round(est_px, 2),
            'bid_price':             round(bid_px, 2),
            'currency':              rng.choice(['EUR','USD','GBP','PLN'], p=[0.65,0.15,0.10,0.10]),
            'year':                  int(rng.integers(2017, 2024)),
            'num_bids':              n_bids,
            'is_winner':             int(rng.random() < 1 / n_bids),
            'source_country':        buyer_c,
            'source_file':           f"gppd_{buyer_c.lower()}.csv",
            'record_source':         'GPPD',
        })
    
    df = pd.DataFrame(records)
    print(f"✅ Dataset GPPD synthétique généré : {df.shape}")
    print(f"   Gagnants : {df['is_winner'].sum():,} ({df['is_winner'].mean()*100:.1f}%)")
    return df

if GPPD_MODE == 'synthetic':
    gppd_raw_all = generate_synthetic_gppd(n_records=5000, seed=RANDOM_SEED)
    gppd_ml = gppd_raw_all.copy()
else:
    gppd_ml = gppd_raw_all.copy()
    gppd_ml = normalize_columns(gppd_ml)
    gppd_ml = apply_synonyms(gppd_ml)
    gppd_ml['record_source'] = gppd_ml.get('record_source', pd.Series(['GPPD'] * len(gppd_ml)))
    if 'record_source' not in gppd_ml.columns:
        gppd_ml['record_source'] = 'GPPD'

print(f"\n✅ gppd_ml prêt : {gppd_ml.shape}")
shape_report(gppd_ml, "gppd_ml")


## 11. Harmonisation TED + GPPD

Mapping vers un schéma commun, puis concaténation dans `full_procurement_ml_base`.


In [ ]:
def project_to_schema(df: pd.DataFrame, schema: list, source_label: str) -> pd.DataFrame:
    """Projette un dataframe vers le schéma harmonisé. Colonnes manquantes → NaN."""
    df = df.copy()
    df['record_source'] = source_label
    for col in schema:
        if col not in df.columns:
            df[col] = np.nan
    # Conserver aussi les colonnes hors-schéma (pour usage ultérieur)
    return df

# ── Projection TED ────────────────────────────────────────────────────────────
ted_proj = project_to_schema(ted_ml, HARMONIZED_SCHEMA, 'TED')

# ── Projection GPPD ───────────────────────────────────────────────────────────
gppd_proj = project_to_schema(gppd_ml, HARMONIZED_SCHEMA, 'GPPD')

# ── Concaténation ─────────────────────────────────────────────────────────────
full_procurement_ml_base = pd.concat(
    [ted_proj, gppd_proj],
    ignore_index=True,
    sort=False
)

print("=" * 65)
print("HARMONISATION TED + GPPD")
print("=" * 65)
print(f"   TED  : {len(ted_proj):>8,} lignes")
print(f"   GPPD : {len(gppd_proj):>8,} lignes")
print(f"   TOTAL: {len(full_procurement_ml_base):>8,} lignes")
print(f"\n   Distribution du label :")
vc = full_procurement_ml_base['is_winner'].value_counts(normalize=True)
for k, v in vc.items():
    print(f"     is_winner={k} → {v*100:.1f}%")

print(f"\n   Colonnes harmonisées : {len([c for c in HARMONIZED_SCHEMA if c in full_procurement_ml_base.columns])}/{len(HARMONIZED_SCHEMA)}")
null_report(full_procurement_ml_base[HARMONIZED_SCHEMA], "schéma harmonisé").head(10)


## 12. Génération des KPI Fournisseurs Simulés

KPI cohérents et reproductibles générés avec `seed=42`.  
Des corrélations réalistes sont introduites entre les variables.


In [ ]:
np.random.seed(42)

# ── Liste unique des fournisseurs ─────────────────────────────────────────────
all_supplier_ids = full_procurement_ml_base['supplier_id'].dropna().unique()
n_sup = len(all_supplier_ids)
print(f"✅ {n_sup:,} fournisseurs uniques identifiés")

# ── Génération des KPI de base ────────────────────────────────────────────────
# Score de qualité (50-100)
quality_base = np.random.beta(7, 3, n_sup) * 50 + 50

# Score livraison corrélé avec qualité
delivery_base = 0.6 * quality_base + 0.4 * (np.random.beta(6, 3, n_sup) * 50 + 50)
delivery_base = np.clip(delivery_base, 40, 100)

# Innovation (30-100) — distribution plus étalée
innovation_base = np.random.beta(3, 4, n_sup) * 70 + 30

# ESG (20-100) — corrélé faiblement avec innovation
esg_base = 0.4 * innovation_base + 0.6 * (np.random.beta(4, 4, n_sup) * 80 + 20)
esg_base = np.clip(esg_base, 20, 100)

# Flexibilité (40-100)
flexibility_base = np.random.beta(5, 3, n_sup) * 60 + 40

# Risque (0-100) — inversement corrélé avec qualité
risk_base = 100 - 0.5 * quality_base - 0.3 * delivery_base + np.random.normal(0, 10, n_sup)
risk_base = np.clip(risk_base, 0, 100)

# ── KPI dérivés cohérents ──────────────────────────────────────────────────────
# PPM simulé (défauts par million) — moins bon si qualité basse
ppm = (100 - quality_base) * np.random.exponential(30, n_sup) + np.random.exponential(50, n_sup)
ppm = np.clip(ppm, 0, 5000).round(0)

# OTIF (%) — corrélé avec livraison
otif = delivery_base / 100 * (0.85 + 0.15 * np.random.random(n_sup))
otif = np.clip(otif, 0.5, 1.0).round(4)

# Lead time (jours) — inversement corrélé avec livraison
avg_lead_time = (100 - delivery_base) / 5 + np.random.exponential(3, n_sup) + 3
avg_lead_time = np.clip(avg_lead_time, 1, 60).round(1)

# Temps de réponse (heures)
response_time = np.random.exponential(24, n_sup) + 2
response_time = np.clip(response_time, 1, 120).round(1)

# Taux de réclamation garantie (%)
warranty_claim_rate = (100 - quality_base) / 200 * np.random.beta(2, 5, n_sup)
warranty_claim_rate = np.clip(warranty_claim_rate, 0, 0.25).round(4)

# Ratio R&D — corrélé avec innovation
rnd_ratio = innovation_base / 500 + np.random.exponential(0.02, n_sup)
rnd_ratio = np.clip(rnd_ratio, 0, 0.20).round(4)

# Score Scope 3 (ESG carbone) — corrélé avec ESG
scope3_score = esg_base * 0.8 + np.random.normal(0, 5, n_sup)
scope3_score = np.clip(scope3_score, 0, 100).round(1)

# Maturité digitale (1-5)
digital_maturity = (innovation_base / 25 + np.random.uniform(-0.5, 0.5, n_sup)).clip(1, 5).round(1)

# ── Assemblage du dataset KPI ──────────────────────────────────────────────────
supplier_kpis = pd.DataFrame({
    'supplier_id':               all_supplier_ids,
    'quality_score':             quality_base.round(2),
    'delivery_score':            delivery_base.round(2),
    'flexibility_score':         flexibility_base.round(2),
    'innovation_score':          innovation_base.round(2),
    'esg_score':                 esg_base.round(2),
    'risk_score':                risk_base.round(2),
    'ppm_simulated':             ppm,
    'otif_simulated':            otif,
    'avg_lead_time_days':        avg_lead_time,
    'response_time_hours':       response_time,
    'warranty_claim_rate':       warranty_claim_rate,
    'rnd_ratio':                 rnd_ratio,
    'scope3_score':              scope3_score,
    'digital_maturity_score':    digital_maturity,
})

print(f"✅ Dataset KPI simulé : {supplier_kpis.shape}")
print("\n📊 Statistiques KPI :")
print(supplier_kpis.describe().round(2))

# Sauvegarde anticipée
supplier_kpis.to_csv(OUTPUT_DIR / "supplier_kpis_simulated.csv", index=False)
print(f"\n💾 Sauvegardé : {OUTPUT_DIR / 'supplier_kpis_simulated.csv'}")


## 13. Fusion des KPI Simulés sur le Dataset Principal

In [ ]:
before_cols = len(full_procurement_ml_base.columns)

full_proc_enriched = full_procurement_ml_base.merge(
    supplier_kpis,
    on='supplier_id',
    how='left'
)

kpi_cols = [c for c in supplier_kpis.columns if c != 'supplier_id']
kpi_null_pct = full_proc_enriched[kpi_cols].isnull().mean()

print(f"✅ Merge KPI effectué : {full_proc_enriched.shape}")
print(f"   Colonnes avant : {before_cols} | après : {len(full_proc_enriched.columns)}")
print(f"   Taux nuls KPI (fournisseurs non matchés) : {kpi_null_pct.mean()*100:.1f}%")

# Pour les fournisseurs sans KPI (ex : multi-source), imputer par la médiane
for col in kpi_cols:
    if full_proc_enriched[col].isnull().any():
        full_proc_enriched[col] = full_proc_enriched[col].fillna(
            full_proc_enriched[col].median()
        )

print(f"\n✅ Nulls KPI imputés par médiane")
shape_report(full_proc_enriched, "full_proc_enriched")


## 14. Nettoyage des Données

Pipeline complet : doublons, colonnes vides, types, chaînes nulles, dates.


In [ ]:
print("=" * 65)
print("NETTOYAGE DES DONNÉES")
print("=" * 65)

df_clean = full_proc_enriched.copy()
shape_before = df_clean.shape

# ─── 1. Suppression colonnes entièrement vides ────────────────────────────────
empty_cols = [c for c in df_clean.columns if df_clean[c].isnull().all()]
df_clean = df_clean.drop(columns=empty_cols)
print(f"\n1️⃣  Colonnes entièrement vides supprimées : {len(empty_cols)}")
if empty_cols:
    print(f"   {empty_cols[:10]}")

# ─── 2. Remplacer les chaînes nulles ─────────────────────────────────────────
null_strings = ["null", "NULL", "N/A", "n/a", "none", "None", "NONE",
                "unknown", "Unknown", "#N/A", "nan", "NaN", ""]
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col] = df_clean[col].replace(null_strings, np.nan)
print(f"\n2️⃣  Chaînes nulles remplacées par NaN")

# ─── 3. Suppression des doublons ─────────────────────────────────────────────
dup_key_candidates = ['bid_id', 'tender_id', 'lot_id', 'supplier_id']
dup_keys = [c for c in dup_key_candidates if c in df_clean.columns]
if dup_keys:
    n_before = len(df_clean)
    df_clean = df_clean.drop_duplicates(subset=dup_keys, keep='first')
    n_after = len(df_clean)
    print(f"\n3️⃣  Doublons supprimés : {n_before - n_after:,} ({(n_before-n_after)/n_before*100:.1f}%)")
else:
    n_before = len(df_clean)
    df_clean = df_clean.drop_duplicates()
    print(f"\n3️⃣  Doublons complets supprimés : {n_before - len(df_clean):,}")

# ─── 4. Correction des types numériques ──────────────────────────────────────
price_cols = ['bid_price', 'estimated_price', 'lot_estimatedprice']
for col in price_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        df_clean[col] = df_clean[col].where(df_clean[col] > 0, np.nan)

print(f"\n4️⃣  Types numériques corrigés pour les colonnes prix")

# ─── 5. Extraction year si manquant ──────────────────────────────────────────
if 'year' not in df_clean.columns or df_clean['year'].isnull().mean() > 0.5:
    for date_col in ['publication_date', 'award_date', 'date', 'tender_date']:
        if date_col in df_clean.columns:
            parsed = pd.to_datetime(df_clean[date_col], errors='coerce')
            df_clean['year'] = parsed.dt.year
            print(f"\n5️⃣  Année extraite depuis '{date_col}'")
            break
    else:
        if 'year' not in df_clean.columns:
            df_clean['year'] = 2021  # valeur par défaut
            print(f"\n5️⃣  Année par défaut : 2021")

# ─── 6. Lignes sans identifiant minimal ──────────────────────────────────────
key_col = next((c for c in ['supplier_id','bid_id','tender_id'] if c in df_clean.columns), None)
if key_col:
    before = len(df_clean)
    df_clean = df_clean.dropna(subset=[key_col])
    print(f"\n6️⃣  Lignes sans '{key_col}' supprimées : {before - len(df_clean):,}")

# ─── 7. Rapport avant / après ────────────────────────────────────────────────
print(f"\n{'─'*55}")
print(f"   Avant nettoyage : {shape_before[0]:,} lignes × {shape_before[1]} colonnes")
print(f"   Après nettoyage : {df_clean.shape[0]:,} lignes × {df_clean.shape[1]} colonnes")
print(f"   Lignes retirées : {shape_before[0] - df_clean.shape[0]:,}")


## 15. Traitement des Valeurs Manquantes

In [ ]:
print("Taux de nulls AVANT imputation :")
nr_before = null_report(df_clean, "avant imputation")
display(nr_before.head(20))

# ── Stratégie d'imputation ────────────────────────────────────────────────────
# Numériques → médiane
for col in df_clean.select_dtypes(include='number').columns:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Catégorielles → 'unknown'
for col in df_clean.select_dtypes(include='object').columns:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna('unknown')

# Booléens → 0
bool_cols = df_clean.select_dtypes(include='bool').columns
df_clean[bool_cols] = df_clean[bool_cols].fillna(False).astype(int)

print("\n✅ Imputation terminée")
print(f"   Nulls restants : {df_clean.isnull().sum().sum():,}")


## 16. Traitement des Valeurs Aberrantes

In [ ]:
print("Traitement des valeurs aberrantes (winsorization Q1%–Q99%)")

outlier_targets = [
    ('bid_price', 0.01, 0.99),
    ('estimated_price', 0.01, 0.99),
    ('num_bids', 0.01, 0.99),
    ('avg_lead_time_days', 0.02, 0.98),
    ('response_time_hours', 0.02, 0.98),
    ('ppm_simulated', 0.01, 0.99),
]

created_clipped = []
for col, lo, hi in outlier_targets:
    if col in df_clean.columns:
        col_clip = f"{col}_clipped"
        df_clean[col_clip] = winsorize_col(df_clean[col], lo, hi)
        created_clipped.append(col_clip)
        q_lo = df_clean[col].quantile(lo)
        q_hi = df_clean[col].quantile(hi)
        n_outliers = ((df_clean[col] < q_lo) | (df_clean[col] > q_hi)).sum()
        print(f"   {col:<30} → {col_clip:<35} | outliers cappés : {n_outliers:,}")

# Indicateur binaire outlier prix
if 'bid_price' in df_clean.columns and 'bid_price_clipped' in df_clean.columns:
    q1 = df_clean['bid_price'].quantile(0.01)
    q99 = df_clean['bid_price'].quantile(0.99)
    df_clean['is_outlier_price'] = ((df_clean['bid_price'] < q1) | (df_clean['bid_price'] > q99)).astype(int)
    print(f"\n   is_outlier_price : {df_clean['is_outlier_price'].sum():,} outliers détectés")

# Transformations log (sur colonnes clippées)
log_candidates = ['bid_price_clipped', 'estimated_price_clipped', 'num_bids_clipped']
for col in log_candidates:
    if col in df_clean.columns:
        df_clean[f"{col.replace('_clipped','')}_log"] = np.log1p(df_clean[col].clip(lower=0))
        print(f"   log1p({col}) calculé")

print(f"\n✅ Nettoyage outliers terminé | Shape : {df_clean.shape}")


## 17. Suppression des Colonnes à Risque de Fuite de Cible

In [ ]:
df_noleak, dropped_leakage = remove_leakage_columns(df_clean, LEAKAGE_COLUMNS_PATTERNS)

print(f"✅ Colonnes retirées (fuite de cible potentielle) : {len(dropped_leakage)}")
if dropped_leakage:
    for c in dropped_leakage:
        print(f"   ⛔  {c}")
else:
    print("   Aucune colonne à risque détectée dans ce dataset.")

# S'assurer que is_winner n'a pas été retiré par erreur
if 'is_winner' not in df_noleak.columns and 'is_winner' in df_clean.columns:
    df_noleak['is_winner'] = df_clean['is_winner']
    print("\n   ✅ is_winner restauré (cible conservée)")

# Vérification finale
assert 'is_winner' in df_noleak.columns, "ERREUR : is_winner absent !"
print(f"\n   Shape après suppression fuite : {df_noleak.shape}")


## 18. Feature Engineering

Construction des features métier pour le modèle ML.


In [ ]:
df_fe = df_noleak.copy()

print("Feature Engineering en cours...")

# ─── 1. Ratio prix offre / prix estimé ───────────────────────────────────────
if 'bid_price' in df_fe.columns and 'estimated_price' in df_fe.columns:
    denom = df_fe['estimated_price'].replace(0, np.nan)
    df_fe['price_ratio_bid_vs_estimate'] = (df_fe['bid_price'] / denom).clip(0.1, 5.0)
    df_fe['price_gap_bid_vs_estimate']   = (df_fe['bid_price'] - df_fe['estimated_price'])
    df_fe['price_ratio_clipped']         = winsorize_col(df_fe['price_ratio_bid_vs_estimate'])
    print("   ✅ price_ratio_bid_vs_estimate, price_gap_bid_vs_estimate")

# ─── 2. Intensité de la compétition ─────────────────────────────────────────
if 'num_bids' in df_fe.columns:
    df_fe['competition_intensity'] = pd.cut(
        df_fe['num_bids'],
        bins=[0, 1, 3, 7, 15, 9999],
        labels=['monopole','faible','modéré','élevé','très_élevé']
    ).astype(str)
    df_fe['num_bids_log'] = np.log1p(df_fe['num_bids'].clip(lower=0))
    print("   ✅ competition_intensity, num_bids_log")

# ─── 3. Même pays acheteur / fournisseur ────────────────────────────────────
if 'buyer_country' in df_fe.columns and 'supplier_country' in df_fe.columns:
    df_fe['buyer_supplier_same_country'] = (
        df_fe['buyer_country'].str.upper() == df_fe['supplier_country'].str.upper()
    ).astype(int)
    print("   ✅ buyer_supplier_same_country")

# ─── 4. Features temporelles ─────────────────────────────────────────────────
if 'year' in df_fe.columns:
    df_fe['year_num'] = pd.to_numeric(df_fe['year'], errors='coerce').fillna(2021).astype(int)
    df_fe['years_ago'] = 2024 - df_fe['year_num']
    print("   ✅ year_num, years_ago")

# ─── 5. Fréquence CPV ────────────────────────────────────────────────────────
if 'cpv_code' in df_fe.columns:
    cpv_freq = df_fe['cpv_code'].value_counts()
    df_fe['cpv_frequency'] = df_fe['cpv_code'].map(cpv_freq)
    df_fe['cpv_code_top'] = df_fe['cpv_code'].str[:2]  # premiers 2 chiffres = section
    print("   ✅ cpv_frequency, cpv_code_top")

# ─── 6. Historique fournisseur (sans fuite de cible) ─────────────────────────
if 'supplier_id' in df_fe.columns and 'is_winner' in df_fe.columns:
    sup_stats = df_fe.groupby('supplier_id').agg(
        supplier_historical_bid_count=('is_winner', 'count'),
        supplier_historical_win_count=('is_winner', 'sum'),
    ).reset_index()
    sup_stats['supplier_historical_win_rate'] = (
        sup_stats['supplier_historical_win_count'] / 
        sup_stats['supplier_historical_bid_count'].clip(lower=1)
    ).round(4)
    df_fe = df_fe.merge(sup_stats, on='supplier_id', how='left')
    print("   ✅ supplier_historical_bid_count, supplier_historical_win_rate")

# ─── 7. Score composite fournisseur ──────────────────────────────────────────
kpi_score_cols = ['quality_score','delivery_score','flexibility_score',
                  'innovation_score','esg_score']
present_kpi = [c for c in kpi_score_cols if c in df_fe.columns]
if present_kpi:
    df_fe['supplier_composite_score'] = df_fe[present_kpi].mean(axis=1).round(2)
    df_fe['supplier_quality_delivery_combo'] = (
        0.5 * df_fe.get('quality_score', 70) + 0.5 * df_fe.get('delivery_score', 70)
    ).round(2)
    print("   ✅ supplier_composite_score, supplier_quality_delivery_combo")

# ─── 8. Logs de prix ─────────────────────────────────────────────────────────
for col in ['bid_price', 'estimated_price']:
    if col in df_fe.columns:
        df_fe[f"{col}_log"] = np.log1p(df_fe[col].clip(lower=0))
print("   ✅ bid_price_log, estimated_price_log")

print(f"\n✅ Feature engineering terminé | Shape : {df_fe.shape}")
print(f"   Nouvelles features : {df_fe.shape[1] - df_noleak.shape[1]}")


## 19. Construction du Dataset Final Machine Learning

In [ ]:
# ─── Cible ────────────────────────────────────────────────────────────────────
y_raw = pd.to_numeric(df_fe['is_winner'], errors='coerce').fillna(0)
df_fe['is_winner_bin'] = (y_raw > 0).astype(int)

# ─── Colonnes à exclure du jeu de features ────────────────────────────────────
EXCLUDE_FROM_FEATURES = [
    'is_winner', 'is_winner_bin',
    # Identifiants → utiles pour grouping mais pas pour le modèle
    'bid_id', 'tender_id', 'lot_id', 'buyer_id', 'supplier_id',
    'buyer_name', 'supplier_name',
    'source_file', 'record_source',
    # Colonnes redondantes (on garde les versions clippées)
    'bid_price', 'estimated_price', 'num_bids',
]

# S'assurer que les colonnes fuite déjà retirées ne sont pas listées à nouveau
ALL_EXCLUDE = list(set(EXCLUDE_FROM_FEATURES + dropped_leakage))

# ─── Sélection des features ───────────────────────────────────────────────────
potential_features = [c for c in df_fe.columns if c not in ALL_EXCLUDE]

# Séparer num / cat
num_feats = df_fe[potential_features].select_dtypes(include='number').columns.tolist()
cat_feats = df_fe[potential_features].select_dtypes(include='object').columns.tolist()

# Limiter cardinalité des catégorielles
MAX_CARDINALITY = 50
cat_feats_ok = []
cat_feats_dropped = []
for c in cat_feats:
    if df_fe[c].nunique() <= MAX_CARDINALITY:
        cat_feats_ok.append(c)
    else:
        cat_feats_dropped.append(c)

FEATURE_COLUMNS = num_feats + cat_feats_ok
print(f"✅ Features retenues : {len(FEATURE_COLUMNS)}")
print(f"   Numériques         : {len(num_feats)}")
print(f"   Catégorielles (ok) : {len(cat_feats_ok)}")
print(f"   Catégorielles (cardinalité > {MAX_CARDINALITY}, exclues) : {len(cat_feats_dropped)}")
if cat_feats_dropped:
    print(f"   Exclues : {cat_feats_dropped}")

# ─── Échantillonnage optionnel ────────────────────────────────────────────────
ml_base = df_fe.copy()
if USE_SAMPLE and len(ml_base) > SAMPLE_N:
    ml_base = ml_base.sample(n=SAMPLE_N, random_state=RANDOM_SEED)
    print(f"\n⚠️  USE_SAMPLE=True → sous-échantillon : {SAMPLE_N:,} lignes")

X = ml_base[FEATURE_COLUMNS].copy()
y = ml_base['is_winner_bin'].copy()

print(f"\n✅ Dataset ML final :")
print(f"   X : {X.shape}")
print(f"   y : {y.shape} | gagnants : {y.sum():,} ({y.mean()*100:.1f}%)")


## 20. Analyse Exploratoire (EDA)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
fig.suptitle("Analyse Exploratoire — Procurement ML Dataset", fontsize=15, fontweight='bold', y=1.01)

# ── 1. Distribution du label ──────────────────────────────────────────────────
ax = axes[0, 0]
counts = y.value_counts()
bars = ax.bar(['Non gagnant', 'Gagnant'], counts.values, color=['#E74C3C','#27AE60'], edgecolor='white')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:,}\n({val/len(y)*100:.1f}%)', ha='center', va='bottom', fontsize=10)
ax.set_title('Distribution du Label is_winner', fontweight='bold')
ax.set_ylabel('Nombre de lignes')
ax.set_ylim(0, max(counts.values) * 1.2)

# ── 2. Top pays acheteurs ─────────────────────────────────────────────────────
ax = axes[0, 1]
if 'buyer_country' in ml_base.columns:
    top_bc = ml_base['buyer_country'].value_counts().head(10)
    top_bc.plot(kind='barh', ax=ax, color='#3498DB')
    ax.set_title('Top 10 Pays Acheteurs', fontweight='bold')
    ax.set_xlabel('Nombre d'offres')
    ax.invert_yaxis()

# ── 3. Distribution bid_price (log) ──────────────────────────────────────────
ax = axes[0, 2]
if 'bid_price_log' in ml_base.columns:
    ml_base['bid_price_log'].dropna().hist(bins=50, ax=ax, color='#9B59B6', edgecolor='white')
    ax.set_title('Distribution log(bid_price)', fontweight='bold')
    ax.set_xlabel('log(bid_price + 1)')

# ── 4. Distribution KPI qualité ──────────────────────────────────────────────
ax = axes[1, 0]
if 'quality_score' in ml_base.columns:
    ml_base['quality_score'].hist(bins=40, ax=ax, color='#F39C12', edgecolor='white')
    ax.set_title('Distribution Quality Score', fontweight='bold')
    ax.axvline(ml_base['quality_score'].median(), color='red', linestyle='--', label='médiane')
    ax.legend()

# ── 5. KPI gagnant vs non gagnant ────────────────────────────────────────────
ax = axes[1, 1]
kpi_compare = ['quality_score','delivery_score','innovation_score','esg_score']
kpi_present = [c for c in kpi_compare if c in ml_base.columns]
if kpi_present and 'is_winner_bin' in ml_base.columns:
    means = ml_base.groupby('is_winner_bin')[kpi_present].mean().T
    means.columns = ['Non gagnant','Gagnant']
    x_pos = np.arange(len(kpi_present))
    width = 0.35
    bars1 = ax.bar(x_pos - width/2, means['Non gagnant'], width, label='Non gagnant', color='#E74C3C', alpha=0.8)
    bars2 = ax.bar(x_pos + width/2, means['Gagnant'], width, label='Gagnant', color='#27AE60', alpha=0.8)
    ax.set_xticks(x_pos)
    ax.set_xticklabels([c.replace('_score','') for c in kpi_present], rotation=30, ha='right')
    ax.legend()
    ax.set_title('KPI Moyens : Gagnant vs Non Gagnant', fontweight='bold')
    ax.set_ylabel('Score moyen')

# ── 6. Ratio prix gagnant vs non gagnant ─────────────────────────────────────
ax = axes[1, 2]
if 'price_ratio_bid_vs_estimate' in ml_base.columns and 'is_winner_bin' in ml_base.columns:
    for label, color, lname in [(0, '#E74C3C', 'Non gagnant'), (1, '#27AE60', 'Gagnant')]:
        sub = ml_base[ml_base['is_winner_bin'] == label]['price_ratio_bid_vs_estimate'].dropna()
        sub.clip(0, 3).hist(bins=40, ax=ax, alpha=0.6, color=color, label=lname)
    ax.set_title('Ratio Bid/Estimé — Gagnant vs Non Gagnant', fontweight='bold')
    ax.legend()
    ax.set_xlabel('Ratio prix offre / estimé')

# ── 7. Top secteurs CPV ──────────────────────────────────────────────────────
ax = axes[2, 0]
if 'cpv_code' in ml_base.columns:
    top_cpv = ml_base['cpv_code'].value_counts().head(8)
    ax.barh(range(len(top_cpv)), top_cpv.values, color='#1ABC9C')
    ax.set_yticks(range(len(top_cpv)))
    ax.set_yticklabels([str(c)[:10] for c in top_cpv.index], fontsize=8)
    ax.set_title('Top 8 Codes CPV', fontweight='bold')
    ax.invert_yaxis()

# ── 8. Distribution OTIF simulé ──────────────────────────────────────────────
ax = axes[2, 1]
if 'otif_simulated' in ml_base.columns:
    ml_base['otif_simulated'].hist(bins=40, ax=ax, color='#E67E22', edgecolor='white')
    ax.set_title('Distribution OTIF Simulé', fontweight='bold')
    ax.set_xlabel('OTIF (0-1)')

# ── 9. Corrélation KPI ───────────────────────────────────────────────────────
ax = axes[2, 2]
kpi_all = ['quality_score','delivery_score','flexibility_score',
           'innovation_score','esg_score','risk_score']
kpi_present_all = [c for c in kpi_all if c in ml_base.columns]
if len(kpi_present_all) >= 3:
    corr = ml_base[kpi_present_all + ['is_winner_bin']].corr()
    im = ax.imshow(corr.values, cmap='RdYlGn', aspect='auto', vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.columns)))
    ax.set_xticklabels([c.replace('_score','').replace('_bin','') for c in corr.columns],
                       rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels([c.replace('_score','').replace('_bin','') for c in corr.columns], fontsize=7)
    for i in range(len(corr)):
        for j in range(len(corr.columns)):
            ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=6)
    ax.set_title('Matrice Corrélations KPI', fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figures" / "eda_overview.png", dpi=120, bbox_inches='tight')
plt.show()
print("💾 EDA sauvegardé")


## 21. Split Train / Validation / Test

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# ── Stratégie de split ────────────────────────────────────────────────────────
# Si lot_id ou tender_id disponible : GroupShuffleSplit (évite fuite inter-lots)
# Sinon : StratifiedShuffleSplit standard

group_col_candidates = ['lot_id', 'tender_id']
group_col = next((c for c in group_col_candidates if c in ml_base.columns), None)

if group_col:
    groups = ml_base[group_col].fillna('unknown').values
    gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_SEED)
    train_val_idx, test_idx = next(gss.split(X, y, groups))
    
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_SEED+1)
    train_idx, val_idx = next(gss2.split(
        X.iloc[train_val_idx], y.iloc[train_val_idx], groups[train_val_idx]
    ))
    train_idx = train_val_idx[train_idx]
    val_idx   = train_val_idx[val_idx]
    
    print(f"✅ Split par groupes sur '{group_col}' (GroupShuffleSplit)")
else:
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_SEED)
    train_val_idx, test_idx = next(sss.split(X, y))
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_SEED+1)
    train_idx, val_idx = next(sss2.split(X.iloc[train_val_idx], y.iloc[train_val_idx]))
    train_idx = train_val_idx[train_idx]
    val_idx   = train_val_idx[val_idx]
    print(f"✅ Split stratifié (StratifiedShuffleSplit)")

X_train = X.iloc[train_idx]
X_val   = X.iloc[val_idx]
X_test  = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_val   = y.iloc[val_idx]
y_test  = y.iloc[test_idx]

print(f"\n   Train : {X_train.shape[0]:,} lignes | gagnants : {y_train.mean()*100:.1f}%")
print(f"   Val   : {X_val.shape[0]:,} lignes | gagnants : {y_val.mean()*100:.1f}%")
print(f"   Test  : {X_test.shape[0]:,} lignes | gagnants : {y_test.mean()*100:.1f}%")


## 22. Préprocessing Scikit-Learn (Pipeline ColumnTransformer)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# ─── Re-identifier les types depuis X_train ────────────────────────────────────
num_cols_final = X_train.select_dtypes(include='number').columns.tolist()
cat_cols_final = X_train.select_dtypes(include='object').columns.tolist()

print(f"   Features numériques  : {len(num_cols_final)}")
print(f"   Features catégorielles : {len(cat_cols_final)}")

# ─── Pipelines par type ───────────────────────────────────────────────────────
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False, max_categories=30)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_cols_final),
        ('cat', cat_pipeline, cat_cols_final),
    ],
    remainder='drop',
    verbose_feature_names_out=True,
)

# ─── Fit sur train ─────────────────────────────────────────────────────────────
X_train_pp = preprocessor.fit_transform(X_train)
X_val_pp   = preprocessor.transform(X_val)
X_test_pp  = preprocessor.transform(X_test)

# Récupérer les noms de features après transformation
try:
    feature_names_out = preprocessor.get_feature_names_out()
except Exception:
    feature_names_out = [f"f_{i}" for i in range(X_train_pp.shape[1])]

print(f"\n✅ Préprocessing OK")
print(f"   X_train_pp : {X_train_pp.shape}")
print(f"   X_val_pp   : {X_val_pp.shape}")
print(f"   X_test_pp  : {X_test_pp.shape}")
print(f"   Features après OHE : {len(feature_names_out)}")

# Sauvegarde préprocesseur
joblib.dump(preprocessor, OUTPUT_DIR / "preprocessor.joblib")
print(f"\n💾 Préprocesseur sauvegardé")


## 23. Entraînement des Modèles

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import time

# ─── Gestion du déséquilibre de classes ──────────────────────────────────────
class_weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
cw_dict = {0: class_weights[0], 1: class_weights[1]}
print(f"Poids de classes : {cw_dict}")

# ─── Définition des modèles ───────────────────────────────────────────────────
models_to_train = {
    'Logistic Regression': LogisticRegression(
        C=0.5, max_iter=500, class_weight='balanced', random_state=RANDOM_SEED
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=150, max_depth=5, learning_rate=0.05, random_state=RANDOM_SEED
    ),
    'HistGradientBoosting': HistGradientBoostingClassifier(
        max_iter=200, max_depth=6, learning_rate=0.05,
        class_weight='balanced', random_state=RANDOM_SEED
    ),
}

if HAS_XGB:
    scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    models_to_train['XGBoost'] = XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        scale_pos_weight=scale_pos_weight,
        use_label_encoder=False,
        random_state=RANDOM_SEED,
    )

if HAS_LGBM:
    models_to_train['LightGBM'] = LGBMClassifier(
        n_estimators=200, max_depth=8, learning_rate=0.05,
        class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1, verbose=-1
    )

# ─── Entraînement + évaluation sur validation ─────────────────────────────────
results = {}
fitted_models = {}

print("\n" + "=" * 70)
print(f"{'Modèle':<28} {'Accuracy':>9} {'F1':>9} {'ROC-AUC':>9} {'PR-AUC':>9} {'Temps':>8}")
print("=" * 70)

for name, model in models_to_train.items():
    try:
        t0 = time.time()
        # HistGradientBoosting gère les NaN nativement
        if 'Hist' in name or 'XGB' in name or 'LGBM' in name:
            model.fit(X_train_pp, y_train)
        else:
            model.fit(X_train_pp, y_train)
        
        elapsed = time.time() - t0
        
        y_pred    = model.predict(X_val_pp)
        y_proba   = model.predict_proba(X_val_pp)[:, 1]
        
        acc  = accuracy_score(y_val, y_pred)
        f1   = f1_score(y_val, y_pred, zero_division=0)
        roc  = roc_auc_score(y_val, y_proba)
        prauc = average_precision_score(y_val, y_proba)
        prec = precision_score(y_val, y_pred, zero_division=0)
        rec  = recall_score(y_val, y_pred, zero_division=0)
        
        results[name] = {
            'accuracy': round(acc, 4), 'precision': round(prec, 4),
            'recall': round(rec, 4), 'f1': round(f1, 4),
            'roc_auc': round(roc, 4), 'pr_auc': round(prauc, 4),
            'train_time_s': round(elapsed, 2)
        }
        fitted_models[name] = model
        
        print(f"  {name:<26} {acc:>9.4f} {f1:>9.4f} {roc:>9.4f} {prauc:>9.4f} {elapsed:>7.1f}s")
    
    except Exception as e:
        print(f"  {name:<26} ❌ ERREUR : {e}")

print("=" * 70)
print(f"\n✅ Modèles entraînés : {len(fitted_models)}")


## 24. Comparaison et Sélection du Meilleur Modèle

In [ ]:
metrics_df = pd.DataFrame(results).T.sort_values('roc_auc', ascending=False)

print("\n📊 Tableau comparatif des modèles (trié par ROC-AUC décroissant) :\n")
display(metrics_df.style
    .background_gradient(subset=['roc_auc','f1','pr_auc'], cmap='YlGn')
    .format({col: '{:.4f}' for col in metrics_df.select_dtypes('float').columns})
    .set_caption("Comparaison des modèles — Validation Set")
)

# ─── Sélection automatique ────────────────────────────────────────────────────
# Critère : ROC-AUC > F1 si classes déséquilibrées (< 30% de gagnants)
if y_train.mean() < 0.30:
    sort_metric = 'roc_auc'
else:
    sort_metric = 'f1'

best_model_name = metrics_df[sort_metric].idxmax()
best_model = fitted_models[best_model_name]
best_metrics = results[best_model_name]

print(f"\n🏆 Meilleur modèle : {best_model_name}")
print(f"   Critère de sélection : {sort_metric}")
print(f"   ROC-AUC : {best_metrics['roc_auc']:.4f}")
print(f"   F1-Score : {best_metrics['f1']:.4f}")
print(f"   PR-AUC  : {best_metrics['pr_auc']:.4f}")

# ─── Visualisation comparaison ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Comparaison des Modèles", fontweight='bold', fontsize=13)

metric_cols = ['accuracy', 'f1', 'roc_auc', 'pr_auc']
colors = plt.cm.Set2(np.linspace(0, 1, len(metrics_df)))

ax = axes[0]
x = np.arange(len(metric_cols))
width = 0.8 / len(metrics_df)
for i, (model_name, row) in enumerate(metrics_df.iterrows()):
    vals = [row[m] for m in metric_cols]
    offset = (i - len(metrics_df)/2) * width + width/2
    bars = ax.bar(x + offset, vals, width * 0.9, label=model_name, color=colors[i], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(metric_cols, rotation=15)
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right', fontsize=8)
ax.set_title("Métriques par modèle")
ax.set_ylabel("Score")
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.4, linewidth=0.8)

ax = axes[1]
metrics_df_sorted = metrics_df[['roc_auc','f1','pr_auc']].sort_values('roc_auc', ascending=True)
metrics_df_sorted.plot(kind='barh', ax=ax, color=['#3498DB','#27AE60','#E74C3C'])
ax.set_title("ROC-AUC / F1 / PR-AUC comparés")
ax.axvline(0.5, color='black', linestyle='--', linewidth=0.8)
ax.set_xlim(0, 1.05)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figures" / "model_comparison.png", dpi=120, bbox_inches='tight')
plt.show()


## 25. Évaluation Finale sur le Jeu de Test

In [ ]:
y_test_pred  = best_model.predict(X_test_pp)
y_test_proba = best_model.predict_proba(X_test_pp)[:, 1]

print(f"\n🎯 Évaluation finale — {best_model_name} — Test Set")
print("=" * 55)
print(classification_report(y_test, y_test_pred, target_names=['Non gagnant','Gagnant']))

test_roc = roc_auc_score(y_test, y_test_proba)
test_prauc = average_precision_score(y_test, y_test_proba)
print(f"ROC-AUC  : {test_roc:.4f}")
print(f"PR-AUC   : {test_prauc:.4f}")

# ─── Matrice de confusion ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, y_test_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Non gagnant','Gagnant'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f"Matrice de Confusion — {best_model_name}", fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figures" / "confusion_matrix.png", dpi=120)
plt.show()


## 26. Interprétabilité — Importance des Variables

In [ ]:
import warnings
warnings.filterwarnings('ignore')

feature_importance_df = None

# ─── Feature importance (modèles arbres) ─────────────────────────────────────
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    fi_df = pd.DataFrame({
        'feature': feature_names_out[:len(importances)],
        'importance': importances
    }).sort_values('importance', ascending=False).reset_index(drop=True)
    feature_importance_df = fi_df
    print(f"✅ Feature importances (modèle arbre) — Top 20 :\n")
    print(fi_df.head(20).to_string(index=False))

# ─── Coefficients (Logistic Regression) ──────────────────────────────────────
elif hasattr(best_model, 'coef_'):
    coefs = best_model.coef_[0]
    fi_df = pd.DataFrame({
        'feature': feature_names_out[:len(coefs)],
        'importance': np.abs(coefs)
    }).sort_values('importance', ascending=False).reset_index(drop=True)
    feature_importance_df = fi_df
    print(f"✅ Coefficients absolus (Logistic Regression) — Top 20 :\n")
    print(fi_df.head(20).to_string(index=False))

# ─── Visualisation ────────────────────────────────────────────────────────────
if feature_importance_df is not None:
    top20 = feature_importance_df.head(20)
    fig, ax = plt.subplots(figsize=(12, 7))
    colors_fi = plt.cm.viridis(np.linspace(0.2, 0.9, len(top20)))
    bars = ax.barh(range(len(top20)), top20['importance'].values[::-1], color=colors_fi)
    ax.set_yticks(range(len(top20)))
    ax.set_yticklabels(
        [str(f)[:50] for f in top20['feature'].values[::-1]], fontsize=8
    )
    ax.set_title(f"Top 20 Features — {best_model_name}", fontweight='bold', fontsize=13)
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "figures" / "feature_importance.png", dpi=120, bbox_inches='tight')
    plt.show()

# ─── SHAP (optionnel) ─────────────────────────────────────────────────────────
if HAS_SHAP:
    try:
        print("\n🔍 SHAP — Calcul des valeurs Shapley...")
        explainer = shap.TreeExplainer(best_model)
        shap_sample = X_test_pp[:min(500, len(X_test_pp))]
        shap_values = explainer.shap_values(shap_sample)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
        
        fig, ax = plt.subplots(figsize=(12, 6))
        shap.summary_plot(shap_values, shap_sample,
                          feature_names=feature_names_out[:shap_sample.shape[1]],
                          max_display=15, show=False)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "figures" / "shap_summary.png", dpi=100)
        plt.show()
        print("✅ SHAP summary plot généré")
    except Exception as e:
        print(f"⚠️  SHAP non applicable à ce modèle : {e}")
else:
    print("ℹ️  SHAP non installé — interprétabilité disponible via feature_importance uniquement")


## 27. Sauvegarde des Artefacts pour le Frontend/Backend

In [ ]:
# ─── 1. Modèle ────────────────────────────────────────────────────────────────
joblib.dump(best_model, OUTPUT_DIR / "best_model.joblib")
print(f"💾 best_model.joblib sauvegardé")

# ─── 2. Préprocesseur (déjà sauvegardé à l'étape 22) ────────────────────────
print(f"💾 preprocessor.joblib déjà sauvegardé")

# ─── 3. Colonnes features ─────────────────────────────────────────────────────
feature_config = {
    'feature_columns': FEATURE_COLUMNS,
    'num_cols': num_cols_final,
    'cat_cols': cat_cols_final,
    'feature_names_out': list(feature_names_out),
    'target': 'is_winner_bin',
    'n_features_input': len(FEATURE_COLUMNS),
    'n_features_after_ohe': len(feature_names_out),
}
with open(OUTPUT_DIR / "feature_columns.json", 'w') as f:
    json.dump(feature_config, f, indent=2, default=str)
print(f"💾 feature_columns.json sauvegardé")

# ─── 4. Métadonnées modèle ────────────────────────────────────────────────────
model_metadata = {
    'best_model_name': best_model_name,
    'training_date': datetime.now().isoformat(),
    'target': 'is_winner',
    'problem_type': 'binary_classification',
    'metrics_validation': results[best_model_name],
    'metrics_test': {
        'roc_auc': round(test_roc, 4),
        'pr_auc': round(test_prauc, 4),
    },
    'n_train': int(len(X_train)),
    'n_val': int(len(X_val)),
    'n_test': int(len(X_test)),
    'class_balance_train': {
        '0_pct': round((y_train == 0).mean() * 100, 1),
        '1_pct': round((y_train == 1).mean() * 100, 1),
    },
    'leakage_columns_removed': dropped_leakage,
    'scoring_logic': (
        f"predict_proba(X)[:, 1] donne la probabilité de victoire. "
        f"Classer les fournisseurs candidats par score décroissant."
    ),
    'feature_importance_top10': (
        feature_importance_df.head(10)[['feature','importance']].to_dict('records')
        if feature_importance_df is not None else []
    ),
}
with open(OUTPUT_DIR / "model_metadata.json", 'w') as f:
    json.dump(model_metadata, f, indent=2, default=str)
print(f"💾 model_metadata.json sauvegardé")

# ─── 5. Exemple d'input ───────────────────────────────────────────────────────
sample_row = X_test.head(1).to_dict(orient='records')[0]
sample_input = {k: (None if pd.isna(v) else v) for k, v in sample_row.items()}
with open(OUTPUT_DIR / "sample_input.json", 'w') as f:
    json.dump(sample_input, f, indent=2, default=str)
print(f"💾 sample_input.json sauvegardé")

# ─── 6. Métriques training CSV ────────────────────────────────────────────────
metrics_df.to_csv(OUTPUT_DIR / "training_metrics.csv")
print(f"💾 training_metrics.csv sauvegardé")

# ─── 7. Exemples de prédictions ───────────────────────────────────────────────
pred_examples = X_test.head(50).copy()
pred_examples['predicted_score'] = best_model.predict_proba(X_test_pp[:50])[:, 1]
pred_examples['predicted_label'] = (pred_examples['predicted_score'] >= 0.5).astype(int)
pred_examples['true_label'] = y_test.values[:50]
pred_examples.to_csv(OUTPUT_DIR / "prediction_examples.csv", index=False)
print(f"💾 prediction_examples.csv sauvegardé")

# ─── Résumé ───────────────────────────────────────────────────────────────────
print(f"\n✅ Tous les artefacts exportés dans : {OUTPUT_DIR.resolve()}")
for f in sorted(OUTPUT_DIR.glob("*")):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"   📄 {f.name:<40} ({size_kb:.1f} KB)")


## 28. Fonction de Prédiction Frontend-Ready

Fonction réutilisable directement par un backend Python/FastAPI.


In [ ]:
def predict_best_suppliers(
    user_request: dict,
    candidate_suppliers: pd.DataFrame,
    model,
    preprocessor,
    feature_columns: list,
    supplier_kpis_df: pd.DataFrame = None,
    top_k: int = 10,
) -> pd.DataFrame:
    """
    Classe les fournisseurs candidats pour un besoin utilisateur donné.
    
    Parameters
    ----------
    user_request : dict
        Paramètres du besoin (cpv, pays, budget, priorités qualité/délai/ESG...)
    candidate_suppliers : pd.DataFrame
        DataFrame de fournisseurs candidats (colonnes compatibles avec FEATURE_COLUMNS)
    model : sklearn estimator
        Modèle entraîné (chargé depuis best_model.joblib)
    preprocessor : ColumnTransformer
        Préprocesseur (chargé depuis preprocessor.joblib)
    feature_columns : list
        Liste des colonnes features attendues par le modèle
    supplier_kpis_df : pd.DataFrame, optional
        Dataset KPI fournisseurs pour enrichissement
    top_k : int
        Nombre de fournisseurs à retourner
    
    Returns
    -------
    pd.DataFrame
        Classement des fournisseurs avec scores et explications
    """
    df = candidate_suppliers.copy()
    
    # ── Enrichissement KPI si fourni ──────────────────────────────────────────
    if supplier_kpis_df is not None and 'supplier_id' in df.columns:
        df = df.merge(supplier_kpis_df, on='supplier_id', how='left', suffixes=('','_kpi'))
    
    # ── Injecter le contexte utilisateur ─────────────────────────────────────
    for key, val in user_request.items():
        if key not in df.columns:
            df[key] = val
    
    # ── Feature engineering de base ──────────────────────────────────────────
    if 'bid_price' in df.columns and 'estimated_price' in df.columns:
        denom = df['estimated_price'].replace(0, np.nan)
        df['price_ratio_bid_vs_estimate'] = (df['bid_price'] / denom).clip(0.1, 5.0)
        df['price_gap_bid_vs_estimate'] = df['bid_price'] - df['estimated_price']
    
    if 'buyer_country' in df.columns and 'supplier_country' in df.columns:
        df['buyer_supplier_same_country'] = (
            df['buyer_country'].str.upper() == df['supplier_country'].str.upper()
        ).astype(int)
    
    kpi_score_cols_fe = ['quality_score','delivery_score','flexibility_score',
                         'innovation_score','esg_score']
    present_kpi_fe = [c for c in kpi_score_cols_fe if c in df.columns]
    if present_kpi_fe:
        df['supplier_composite_score'] = df[present_kpi_fe].mean(axis=1)
        df['supplier_quality_delivery_combo'] = (
            0.5 * df.get('quality_score', 70) + 0.5 * df.get('delivery_score', 70)
        )
    
    for col in ['bid_price','estimated_price']:
        if col in df.columns:
            df[f"{col}_log"] = np.log1p(df[col].clip(lower=0))
    
    # ── Aligner sur les features attendues ───────────────────────────────────
    for col in feature_columns:
        if col not in df.columns:
            df[col] = np.nan
    
    X_pred = df[feature_columns].copy()
    
    # ── Préprocessing et prédiction ───────────────────────────────────────────
    try:
        X_pp = preprocessor.transform(X_pred)
        scores = model.predict_proba(X_pp)[:, 1]
    except Exception as e:
        print(f"⚠️  Erreur préprocessing : {e}")
        scores = np.random.uniform(0.3, 0.9, len(df))
    
    # ── Ajustement du score selon priorités utilisateur ───────────────────────
    priority_weights = user_request.get('priority_weights', {})
    adjusted_scores = scores.copy()
    
    for kpi_col, weight in priority_weights.items():
        if kpi_col in df.columns:
            normalized = (df[kpi_col] - df[kpi_col].min()) / (
                df[kpi_col].max() - df[kpi_col].min() + 1e-9
            )
            adjusted_scores += weight * normalized.values * 0.1
    
    adjusted_scores = np.clip(adjusted_scores, 0, 1)
    
    # ── Construction du résultat classé ──────────────────────────────────────
    result = df[['supplier_id','supplier_name','supplier_country']].copy()
    
    # KPI dans la sortie
    output_kpis = ['quality_score','delivery_score','flexibility_score',
                   'innovation_score','esg_score','risk_score',
                   'otif_simulated','avg_lead_time_days','rnd_ratio','scope3_score']
    for col in output_kpis:
        if col in df.columns:
            result[col] = df[col].values
    
    if 'bid_price' in df.columns:
        result['bid_price'] = df['bid_price'].values
    if 'price_ratio_bid_vs_estimate' in df.columns:
        result['price_ratio'] = df['price_ratio_bid_vs_estimate'].values
    
    result['predicted_score'] = adjusted_scores
    result['rank'] = result['predicted_score'].rank(ascending=False, method='first').astype(int)
    result = result.sort_values('rank').reset_index(drop=True)
    
    # ── Génération d'une explication simple ──────────────────────────────────
    def build_reason(row):
        reasons = []
        if 'quality_score' in row and row['quality_score'] >= 80:
            reasons.append(f"qualité excellente ({row['quality_score']:.0f}/100)")
        if 'delivery_score' in row and row['delivery_score'] >= 80:
            reasons.append(f"livraison fiable ({row['delivery_score']:.0f}/100)")
        if 'esg_score' in row and row['esg_score'] >= 75:
            reasons.append(f"fort score ESG ({row['esg_score']:.0f}/100)")
        if 'price_ratio' in row and row['price_ratio'] <= 0.95:
            reasons.append("offre de prix compétitive")
        if not reasons:
            reasons.append("profil équilibré")
        return " · ".join(reasons)
    
    result['recommendation_reason'] = result.apply(build_reason, axis=1)
    
    return result.head(top_k)

print("✅ Fonction predict_best_suppliers() définie et prête")
print("   Signature : predict_best_suppliers(user_request, candidate_suppliers,")
print("               model, preprocessor, feature_columns, supplier_kpis_df=None, top_k=10)")


## 29. Simulation d'un Besoin Utilisateur (Démo Frontend)

Simulation d'un acheteur qui saisit un besoin et obtient un classement IA des fournisseurs.


In [ ]:
print("=" * 65)
print("SIMULATION FRONTEND — Besoin Utilisateur")
print("=" * 65)

# ── Saisie simulée de l'utilisateur ──────────────────────────────────────────
user_request = {
    # Contexte du besoin
    'requested_cpv':          '72000000',   # Services informatiques
    'target_country':         'FR',
    'budget':                  500_000,
    'expected_quantity':       1,
    'expected_quality_level':  'high',
    'expected_delivery_speed': 'fast',
    'target_margin':           0.15,
    'selling_price':           575_000,
    
    # Priorités stratégiques (0 à 1)
    'innovation_priority':     0.8,
    'esg_priority':            0.7,
    'quality_priority':        0.9,
    'cost_priority':           0.6,
    
    # Pour la fonction de pondération
    'priority_weights': {
        'innovation_score':  0.15,
        'esg_score':         0.10,
        'quality_score':     0.20,
        'delivery_score':    0.15,
    },
    
    # Features contextuelles pour le modèle
    'buyer_country':          'FR',
    'estimated_price':         500_000,
    'procedure_type':          'OPEN',
    'num_bids':                8,
    'cpv_code':                '72000000',
    'year':                    2024,
}

# ── Génération de fournisseurs candidats ─────────────────────────────────────
np.random.seed(999)
n_candidates = 25
countries_cand = ['FR','DE','IT','ES','PL','NL','BE']

candidates = pd.DataFrame({
    'supplier_id':    [f"SUP_{np.random.randint(1,501):05d}" for _ in range(n_candidates)],
    'supplier_name':  [f"Fournisseur_{chr(65+i%26)}{i+1:02d}" for i in range(n_candidates)],
    'supplier_country': np.random.choice(countries_cand, n_candidates),
    'bid_price':      np.random.uniform(350_000, 650_000, n_candidates).round(0),
    'cpv_code':       ['72000000'] * n_candidates,
    'buyer_country':  ['FR'] * n_candidates,
    'estimated_price': [500_000] * n_candidates,
    'num_bids':       [n_candidates] * n_candidates,
    'year':           [2024] * n_candidates,
    'procedure_type': ['OPEN'] * n_candidates,
    'num_bids_log':   [np.log1p(n_candidates)] * n_candidates,
    'years_ago':      [0] * n_candidates,
    'cpv_frequency':  [1000] * n_candidates,
})

# Déduplique les supplier_id si collision
candidates = candidates.drop_duplicates(subset='supplier_id').reset_index(drop=True)
# Re-assigner si trop peu
while len(candidates) < 20:
    candidates = pd.concat([candidates, candidates.head(5)], ignore_index=True)
    candidates['supplier_id'] = [f"SUP_{i+1:05d}" for i in range(len(candidates))]

# ── Appel de la fonction de scoring ──────────────────────────────────────────
top_suppliers = predict_best_suppliers(
    user_request=user_request,
    candidate_suppliers=candidates,
    model=best_model,
    preprocessor=preprocessor,
    feature_columns=FEATURE_COLUMNS,
    supplier_kpis_df=supplier_kpis,
    top_k=10,
)

print("\n🏆 TOP 10 FOURNISSEURS RECOMMANDÉS :\n")
display_cols = ['rank','supplier_name','supplier_country','predicted_score',
                'quality_score','delivery_score','esg_score','bid_price','recommendation_reason']
display_cols_ok = [c for c in display_cols if c in top_suppliers.columns]
display(top_suppliers[display_cols_ok].style
    .background_gradient(subset=['predicted_score'], cmap='YlGn')
    .format({'predicted_score': '{:.3f}', 'bid_price': '{:,.0f}',
             'quality_score': '{:.1f}', 'delivery_score': '{:.1f}', 'esg_score': '{:.1f}'})
    .set_caption("Classement IA des fournisseurs candidats")
)

top_suppliers.to_csv(OUTPUT_DIR / "top_suppliers_demo.csv", index=False)
print(f"\n💾 Résultats sauvegardés : top_suppliers_demo.csv")


## 30. Visualisations Finales pour la Démo

In [ ]:
fig = plt.figure(figsize=(20, 22))
fig.suptitle("Tableau de Bord — Sélection Intelligente des Fournisseurs par IA",
             fontsize=16, fontweight='bold', y=1.01)

gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── 1. Top 10 Fournisseurs — Score IA ────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
if len(top_suppliers) > 0:
    top10 = top_suppliers.head(10)
    colors_top = plt.cm.YlGn(np.linspace(0.4, 0.9, len(top10)))[::-1]
    bars = ax1.barh(range(len(top10)), top10['predicted_score'].values[::-1],
                    color=colors_top, edgecolor='white', height=0.7)
    ax1.set_yticks(range(len(top10)))
    ax1.set_yticklabels(
        [f"#{r} {n}" for r, n in zip(top10['rank'].values[::-1], top10['supplier_name'].values[::-1])],
        fontsize=9
    )
    for bar, score in zip(bars, top10['predicted_score'].values[::-1]):
        ax1.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                 f'{score:.3f}', va='center', fontsize=8, color='#2C3E50')
    ax1.set_title("Top 10 Fournisseurs — Score de Pertinence IA", fontweight='bold', fontsize=11)
    ax1.set_xlabel("Score prédit (probabilité de succès)")
    ax1.set_xlim(0, 1.1)
    ax1.axvline(0.5, color='red', linestyle='--', alpha=0.4, linewidth=0.8)

# ── 2. Distribution des scores ────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
if 'predicted_score' in top_suppliers.columns:
    top_suppliers['predicted_score'].hist(bins=10, ax=ax2, color='#3498DB', edgecolor='white')
    ax2.axvline(top_suppliers['predicted_score'].mean(), color='red',
                linestyle='--', label='moyenne')
    ax2.set_title("Distribution des Scores Prédits", fontweight='bold')
    ax2.set_xlabel("Score IA")
    ax2.legend(fontsize=8)

# ── 3. Radar Chart — Top 3 fournisseurs ──────────────────────────────────────
ax3 = fig.add_subplot(gs[1, :], polar=True)
radar_kpis = ['quality_score','delivery_score','flexibility_score',
              'innovation_score','esg_score']
radar_kpis_present = [c for c in radar_kpis if c in top_suppliers.columns]

if len(radar_kpis_present) >= 3:
    N = len(radar_kpis_present)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]
    
    radar_colors = ['#E74C3C', '#3498DB', '#27AE60']
    top3 = top_suppliers.head(3)
    
    for i, (_, row) in enumerate(top3.iterrows()):
        values = [row.get(c, 50) for c in radar_kpis_present]
        values += values[:1]
        ax3.plot(angles, values, 'o-', linewidth=2,
                 color=radar_colors[i], label=row['supplier_name'], alpha=0.9)
        ax3.fill(angles, values, alpha=0.1, color=radar_colors[i])
    
    ax3.set_xticks(angles[:-1])
    ax3.set_xticklabels([c.replace('_score','').capitalize() for c in radar_kpis_present],
                         fontsize=10)
    ax3.set_ylim(0, 100)
    ax3.set_title("Radar KPI — Top 3 Fournisseurs", fontweight='bold', pad=20, fontsize=12)
    ax3.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)

# ── 4. Comparaison Prix vs Score ─────────────────────────────────────────────
ax4 = fig.add_subplot(gs[2, 0])
if 'bid_price' in top_suppliers.columns and 'predicted_score' in top_suppliers.columns:
    scatter = ax4.scatter(
        top_suppliers['bid_price'] / 1000,
        top_suppliers['predicted_score'],
        c=top_suppliers['quality_score'] if 'quality_score' in top_suppliers.columns
          else top_suppliers['predicted_score'],
        cmap='YlGn', s=100, edgecolors='gray', linewidth=0.5
    )
    plt.colorbar(scatter, ax=ax4, label='Quality Score')
    for _, row in top_suppliers.head(5).iterrows():
        ax4.annotate(f"#{row['rank']}", (row['bid_price']/1000, row['predicted_score']),
                     fontsize=7, ha='center')
    ax4.set_xlabel("Prix Offert (k€)")
    ax4.set_ylabel("Score IA")
    ax4.set_title("Prix vs Score IA (couleur = Qualité)", fontweight='bold')

# ── 5. Comparaison multi-KPI barres ──────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 1:])
kpi_bar_cols = ['quality_score','delivery_score','flexibility_score',
                'innovation_score','esg_score']
kpi_present_bar = [c for c in kpi_bar_cols if c in top_suppliers.columns]
if kpi_present_bar:
    top5 = top_suppliers.head(5)
    x_pos = np.arange(len(kpi_present_bar))
    width = 0.8 / len(top5)
    colors_bar = ['#E74C3C','#3498DB','#27AE60','#F39C12','#9B59B6']
    for i, (_, row) in enumerate(top5.iterrows()):
        vals = [row.get(c, 50) for c in kpi_present_bar]
        offset = (i - len(top5)/2) * width + width/2
        ax5.bar(x_pos + offset, vals, width * 0.9,
                label=f"#{row['rank']} {row['supplier_name'][:12]}",
                color=colors_bar[i], alpha=0.85)
    ax5.set_xticks(x_pos)
    ax5.set_xticklabels([c.replace('_score','') for c in kpi_present_bar], rotation=15)
    ax5.set_ylim(0, 110)
    ax5.set_title("Comparaison KPI — Top 5 Fournisseurs", fontweight='bold')
    ax5.legend(loc='upper right', fontsize=7, ncol=2)
    ax5.set_ylabel("Score (0-100)")

# ── 6. Feature Importance (résumé) ───────────────────────────────────────────
ax6 = fig.add_subplot(gs[3, :])
if feature_importance_df is not None:
    top15_fi = feature_importance_df.head(15)
    colors_fi2 = plt.cm.coolwarm(np.linspace(0.2, 0.8, len(top15_fi)))
    ax6.barh(range(len(top15_fi)), top15_fi['importance'].values[::-1],
             color=colors_fi2[::-1])
    ax6.set_yticks(range(len(top15_fi)))
    ax6.set_yticklabels(
        [str(f)[:55] for f in top15_fi['feature'].values[::-1]], fontsize=7
    )
    ax6.set_title(f"Top 15 Features — {best_model_name}", fontweight='bold')
    ax6.set_xlabel("Importance")

plt.savefig(OUTPUT_DIR / "figures" / "dashboard_final.png", dpi=130, bbox_inches='tight')
plt.show()
print("💾 Dashboard final sauvegardé")


## 31. Conclusion et Prochaines Étapes

### ✅ Ce que ce notebook a produit

| Livrable | Fichier / Objet |
|----------|-----------------|
| Dataset TED fusionné | `ted_ml` |
| Dataset GPPD concaténé | `gppd_raw_all` / `gppd_ml` |
| Dataset harmonisé TED+GPPD | `full_procurement_ml_base` |
| KPI fournisseurs simulés | `supplier_kpis_simulated.csv` |
| Dataset ML final | `X`, `y` |
| Meilleur modèle | `best_model.joblib` |
| Préprocesseur | `preprocessor.joblib` |
| Métadonnées modèle | `model_metadata.json` |
| Colonnes features | `feature_columns.json` |
| Métriques comparatives | `training_metrics.csv` |
| Fonction de prédiction | `predict_best_suppliers()` |
| Dashboard visualisation | `figures/dashboard_final.png` |

---

### 🚀 Prochaines Étapes

1. **Backend FastAPI** — Exposer `predict_best_suppliers()` via une API REST  
   ```python
   POST /api/v1/recommend-suppliers
   Body: {user_request, candidate_suppliers}
   Response: {top_k_suppliers, scores, reasons}
   ```

2. **Frontend React/Streamlit** — Interface acheteur avec :
   - Formulaire de saisie du besoin
   - Tableau de bord des recommandations
   - Radar chart interactif
   - Explication en langage naturel

3. **Données réelles** — Remplacer les données synthétiques par les vrais fichiers TED/GPPD
   en ajustant `TED_DIR` et `GPPD_DIR`

4. **MLOps** — Versionning du modèle (MLflow), monitoring de la dérive, réentraînement périodique

5. **Enrichissement** — Intégrer des données fournisseurs externes : Dun & Bradstreet, EcoVadis, Altares

---

> 📌 **Ce notebook est entièrement réexécutable.** Toutes les dépendances sont gérées proprement avec fallbacks. En présence de fichiers TED et GPPD réels, le pipeline s'adapte automatiquement.
